In [13]:
'''
Here we find out the embeddings using arcface
script 2
~ ssmtcj
'''

'''
Why InsightFace (ArcFace) ?
    i) Very robust to varying image quality

    ii)Well-suited for matching high-quality images against CCTV-like frames

    iii)Outputs a 512-dimensional vector per face

    iv) Works offline in Colab
'''

'\nWhy InsightFace (ArcFace) ?\n    i) Very robust to varying image quality\n\n    ii)Well-suited for matching high-quality images against CCTV-like frames\n\n    iii)Outputs a 512-dimensional vector per face\n\n    iv) Works offline in Colab\n'

**Dependencies**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install InsightFace
!pip install insightface onnxruntime

import pandas as pd
import numpy as np
import cv2
import os
import pickle
from insightface.app import FaceAnalysis

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp311-cp311-linux_x86_64.whl size=1060440 sha256=df3a4f8df67b95b79dd10b1437c65975ad02d0199ca9bffd858ac5c4cc0cf2e6
  Stored in directory: /root/.cache/pip/wheels/27/d8/22/f52d858d16cd06e7b2e6aad34a1777dcfaf000be833bbf8146
Successfully built insightface


In [14]:
'''
#DEBUGGING SCRIPT

import pandas as pd
import cv2
import matplotlib.pyplot as plt

# Load CSV
csv_path = "/content/drive/MyDrive/Colab Notebooks/RP/dataset.csv"
df = pd.read_csv(csv_path)

# Function to display an image with title
def show_image(img_path, title=None):
    img = cv2.imread(img_path)
    if img is None:
        print(f"❌ Could not load image: {img_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB for plt
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

# Display all images with their name and id as title
for idx, row in df.iterrows():
    show_image(row['image'], title=f"Name: {row['name']} | ID: {row['id']}")


import os

image_folder = "/content/drive/MyDrive/Colab Notebooks/RP/images"
files = os.listdir(image_folder)
print("Files in images folder:")
for f in files:
    print(f)

'''

Output hidden; open in https://colab.research.google.com to view.

In [24]:
# Initialize face analysis model
app = FaceAnalysis(providers=['CPUExecutionProvider'])  # or 'CUDAExecutionProvider' if GPU
app.prepare(ctx_id=0, det_size=(480, 480))

embeddings = []
metadata = []

for idx, row in df.iterrows():
    img_path = row['image']
    name = row['name']
    roll_id = row['id']

    # Read image
    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Could not load {img_path}")
        continue

    # Detect faces and get embeddings
    faces = app.get(img)
    if len(faces) == 0:
        print(f"⚠️ No face found in {img_path}")
        continue

    # Take the first detected face
    emb = faces[0].embedding  # 512-d vector
    embeddings.append(emb)
    metadata.append({
        "name": name,
        "id": roll_id,
        "image_path": img_path
    })

# Convert to numpy array
embeddings = np.array(embeddings, dtype='float32')

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (480, 480)


In [25]:

# Save to pickle in Drive
output_path = "/content/drive/MyDrive/Colab Notebooks/RP/embeddings.pkl"
with open(output_path, 'wb') as f:
    pickle.dump({"embeddings": embeddings, "metadata": metadata}, f)

print(f"✅ Embeddings saved to {output_path}")
print(f"Shape: {embeddings.shape}")


✅ Embeddings saved to /content/drive/MyDrive/Colab Notebooks/RP/embeddings.pkl
Shape: (10, 512)
